# Модуль 4. Паттерны: Game Loop, Update Method, Component

**Курс:** Объектно-ориентированное программирование для разработки компьютерных игр  
**Модуль:** 4 — Паттерны игрового цикла и компонентов  
**Теория:** 8 ч | **Практика:** 10 ч | **Самостоятельная работа:** 8 ч

---

## О чём этот модуль

До этого модуля мы строили архитектуру игры как **набор объектов**, но не касались самого главного вопроса: **как эти объекты оживают**. Кто вызывает `update()`? С какой частотой? Что происходит между кадрами? Как обеспечить одинаковое поведение игры на быстром и медленном компьютере?

Этот модуль отвечает на эти вопросы через три классических паттерна из книги Роберта Нистрома «Game Programming Patterns»:

- **Game Loop** — сердце любой игры. Отвечает за то, чтобы игровой мир обновлялся и отрисовывался с постоянной частотой.
- **Update Method** — способ обновления множества объектов единообразно, без гигантских списков `if`.
- **Component** — способ собирать игровые объекты из переиспользуемых частей, избегая комбинаторного взрыва наследования.

Эти три паттерна — фундамент для всего, что будет дальше. В модуле 7 мы придём к **ECS** (Entity-Component-System) — современной архитектуре, которая строится на этих же идеях.

## Где мы находимся в проекте

После модуля 3 проект «Тайна исчезнувшего дива» имеет:

| Слой | Что есть |
|---|---|
| **Сущности** | `Detective`, `Div`, `Location`, `Clue` |
| **Абстракции** | `GameObject` (ABC), `Movable`/`Drawable`/`Collidable` (Protocol) |
| **Оркестрация** | `Game`, системы (`InvestigationSystem`, `DialogueSystem`, `MovementSystem`) |
| **Инфраструктура** | `UI`, `InputHandler`, `Renderer`, `GameState`, `LoggingMixin` |

**Чего не хватает:** игровой цикл пока «ручной» — `Game.run()` содержит `while` с ожиданием ввода. Настоящий игровой цикл работает с **временем**, а не с вводом. В этом модуле мы это исправим и добавим **компоненты** для гибкости сущностей.

## Что вы изучите

1. Классический Game Loop и его проблемы.
2. Fixed timestep vs variable timestep.
3. Проблема «спирали смерти» и как её решить.
4. Паттерн Update Method: единообразное обновление объектов.
5. Паттерн Component: композиция поведения вместо наследования.
6. Введение в ECS — задел на модуль 7.
7. Интеграция с Pygame.

## Содержание
1. Зачем нужен игровой цикл
2. Классический Game Loop
3. Fixed timestep: стабильная физика
4. Variable timestep и его проблемы
5. Спираль смерти
6. Интерполяция для плавности
7. Update Method
8. Component: композиция поведения
9. Component в стиле dataclass
10. Переход к ECS
11. Интеграция с Pygame
12. Итоги и литература

In [ ]:
import sys
print(f"Python: {sys.version}")
assert sys.version_info >= (3, 10), "Требуется Python 3.10+"

# Проверка Pygame (для примеров игрового цикла)
try:
    import pygame
    print(f"pygame: {pygame.version.ver}")
except ImportError:
    print("pygame не установлен — примеры с ним не запустятся.")
    print("Установите: pip install pygame")

---
# 1. Зачем нужен игровой цикл

## 1.1. Отличие игры от обычной программы

**Обычная программа** (калькулятор, редактор) работает по принципу «запрос → ответ». Пользователь нажал кнопку — программа отреагировала. Ничего не происходит, пока пользователь не взаимодействует.

**Игра** — принципиально другое. Даже если игрок ничего не делает, мир продолжает меняться:
- враги патрулируют;
- снаряды летят;
- анимации проигрываются;
- музыка играет;
- таймеры тикают.

Игра — это **симуляция**, которая работает непрерывно, вне зависимости от ввода игрока.

## 1.2. Что такое кадр

**Кадр** (frame) — один цикл обновления игры. В каждом кадре игра:
1. Обрабатывает ввод.
2. Обновляет состояние мира (`update`).
3. Отрисовывает мир (`draw`).

Частота кадров измеряется в **FPS** (frames per second). Стандарт — **60 FPS**, то есть 60 кадров в секунду, один кадр ≈ 16.6 мс.

**Для игр 60 FPS** — это баланс между плавностью и ресурсами. Выше 60 FPS человеческий глаз обычно не различает. Ниже 30 FPS — заметны рывки.

## 1.3. Проблема: асинхронность железа

Разные компьютеры работают с разной скоростью. Один кадр может занять:
- 1 мс на мощном ПК;
- 20 мс на старом ноутбуке.

Если не управлять временем, игра будет работать **с разной скоростью** на разных машинах:
- на быстром ПК снаряд летит мгновенно;
- на медленном — медленно.

**Решение:** игровой цикл, который работает с **временем** (дельтами), а не с количеством итераций.

## 1.4. Что даст игровой цикл проекту

Сейчас в «Тайне исчезнувшего дива» игровой цикл примитивен: `while energy > 0: input(); handle()`. Это **цикл по событиям**, а не по времени.

С настоящим игровым циклом:
- появятся **таймеры** (например, время до следующего хода Кузи);
- появятся **анимации** (плавное появление текста, мигание курсора);
- появится **фоновая логика** (случайные события, погода, дыхание мира);
- проект будет готов к переходу на Pygame (модуль 4 — последний текстовый; дальше — Pygame).

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns», глава [Game Loop](https://gameprogrammingpatterns.com/game-loop.html) — каноническое объяснение с примерами на C++.
- [Real Python: Pygame Tutorial](https://realpython.com/pygame-a-primer/) — введение в Pygame с примерами игрового цикла.
- [Pygame documentation: pygame.time.Clock](https://www.pygame.org/docs/ref/time.html) — официальная документация по управлению временем.

---
# 2. Классический Game Loop

## 2.1. Суть паттерна

**Game Loop** — это бесконечный цикл, который:
1. Обрабатывает ввод.
2. Обновляет состояние.
3. Отрисовывает.
4. Повторяет.

Псевдокод:
```python
while game_is_running:
    process_input()
    update()
    render()
```

**Почему это работает:** цикл крутится так быстро, как позволяет железо. Если железо быстрое — много кадров в секунду; если медленное — меньше. Проблема в том, что без управления временем игра будет работать с разной скоростью на разных машинах.

## 2.2. Простейший игровой цикл на Python

Пример без библиотек — с ручным измерением времени:

In [ ]:
import time


def simple_game_loop(duration_seconds: float = 0.5) -> None:
    """Простейший игровой цикл: обновление + отрисовка.

    Работает пока не истечёт duration_seconds.
    Демонстрирует проблему: скорость зависит от железа.
    """
    start = time.perf_counter()
    frames = 0
    while time.perf_counter() - start < duration_seconds:
        # update(dt) — здесь dt не используется
        # draw() — здесь ничего не рисуем
        frames += 1
    elapsed = time.perf_counter() - start
    print(f"Кадров: {frames}, время: {elapsed:.3f} с, FPS: {frames / elapsed:.0f}")


print("=== Простейший игровой цикл без управления временем ===")
simple_game_loop(0.3)

## 2.3. Проблема: скорость зависит от железа

Обратите внимание: цикл без управления временем выполняется **как можно быстрее**. На мощном ПК — миллионы итераций в секунду; на слабом — тысячи. Если в `update()` есть движение, оно будет происходить с разной скоростью.

## 2.4. Наивное решение: фиксированная задержка

Первое, что приходит в голову — добавить `time.sleep()` в конце каждого кадра:

```python
FRAME_TIME = 1.0 / 60  # 60 FPS
while running:
    frame_start = time.perf_counter()
    process_input()
    update()
    render()
    elapsed = time.perf_counter() - frame_start
    if elapsed < FRAME_TIME:
        time.sleep(FRAME_TIME - elapsed)
```

**Проблема этого подхода:** если `update()` занял больше `FRAME_TIME`, мы **пропускаем** кадр. Игра замедляется, потому что цикл не может догнать время.

Демонстрация:

In [ ]:
def loop_with_sleep(duration_seconds: float = 0.3,
                    target_fps: int = 60,
                    work_time: float = 0.0) -> None:
    """Цикл с фиксированной задержкой и эмуляцией 'тяжёлого' update."""
    frame_time = 1.0 / target_fps
    start = time.perf_counter()
    frames = 0
    while time.perf_counter() - start < duration_seconds:
        frame_start = time.perf_counter()
        if work_time > 0:
            time.sleep(work_time)  # эмуляция работы
        elapsed = time.perf_counter() - frame_start
        if elapsed < frame_time:
            time.sleep(frame_time - elapsed)
        frames += 1
    real_time = time.perf_counter() - start
    print(f"Целевой FPS: {target_fps}, факт: {frames / real_time:.0f}")


print("=== Лёгкий update ===")
loop_with_sleep(0.3, target_fps=60, work_time=0.001)

print("=== Тяжёлый update (10 мс на кадр) ===")
loop_with_sleep(0.3, target_fps=60, work_time=0.010)

**Вывод:** при тяжёлом `update()` мы не успеваем сделать 60 кадров за секунду. Игра замедляется. Это значит, что **скорость игры зависит от производительности** — плохо.

## 2.5. Решение: разделить update и render

Настоящее решение — **fixed timestep**: обновлять состояние с постоянным шагом, независимо от частоты отрисовки. Об этом — следующий раздел.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Game Loop](https://gameprogrammingpatterns.com/game-loop.html).
- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/) — классическая статья, объясняющая fixed timestep.

---
# 3. Fixed timestep: стабильная физика

## 3.1. Идея

**Fixed timestep** (фиксированный шаг) — это подход, при котором `update()` вызывается с **постоянным** `dt`, независимо от реального времени:

```python
FIXED_DT = 1.0 / 60  # 16.6 мс
accumulator = 0.0

while running:
    frame_time = measure_frame_time()
    accumulator += frame_time

    while accumulator >= FIXED_DT:
        update(FIXED_DT)  # всегда одно и то же dt
        accumulator -= FIXED_DT

    render()  # отрисовка — с частотой железа
```

**Ключевая идея:** мы накапливаем реальное время в **аккумуляторе**. Пока накопленного времени хватает на шаг физики — выполняем `update` с фиксированным `dt`.

## 3.2. Почему это важно

**Физика нелинейна.** Если `dt` меняется, результаты симуляции будут разными. Пример:

```
Позиция += скорость * dt
```

При `dt = 0.01` и при `dt = 0.1` результат будет **разным** из-за накопления ошибок округления. Прыжок в платформере будет выше или ниже в зависимости от FPS. Снаряд пролетит сквозь стену на быстром кадре и остановится на медленном.

**Fixed timestep** гарантирует, что физика всегда считается одинаково. Игра ведёт себя предсказуемо на любом железе.

## 3.3. Реализация

Ниже — учебная реализация fixed timestep без Pygame. Мы эмулируем переменное время кадра, чтобы показать, как аккумулятор сглаживает его.

In [ ]:
class FixedTimestepLoop:
    """Игровой цикл с фиксированным шагом физики.

    Демонстрирует:
    - accumulator: накопление реального времени;
    - fixed_dt: постоянный шаг для update;
    - переменную частоту render.
    """

    def __init__(self, fixed_dt: float = 1.0 / 60, max_steps: int = 5):
        self.fixed_dt = fixed_dt
        self.max_steps = max_steps
        self.accumulator = 0.0
        self.update_count = 0
        self.render_count = 0

    def tick(self, frame_time: float, update, render) -> None:
        """Один проход цикла.

        frame_time — сколько реально прошло с прошлого кадра.
        update — функция обновления, принимает fixed_dt.
        render — функция отрисовки.
        """
        self.accumulator += frame_time

        steps = 0
        while self.accumulator >= self.fixed_dt and steps < self.max_steps:
            update(self.fixed_dt)
            self.accumulator -= self.fixed_dt
            self.update_count += 1
            steps += 1

        render()
        self.render_count += 1


# Демонстрация: эмулируем переменное время кадра
def demo_fixed_timestep():
    loop = FixedTimestepLoop(fixed_dt=1.0 / 60)
    times = [0.005, 0.008, 0.020, 0.015, 0.012, 0.030, 0.010, 0.006]
    updates_seen = []

    def update(dt):
        updates_seen.append(dt)

    def render():
        pass

    for t in times:
        loop.tick(t, update, render)

    print(f"Кадров render: {loop.render_count}")
    print(f"Вызовов update: {loop.update_count}")
    print(f"Все dt одинаковые: {len(set(updates_seen)) == 1}")
    print(f"Фиксированный dt: {updates_seen[0]:.4f}")


print("=== Fixed timestep ===")
demo_fixed_timestep()

**Обратите внимание:** все вызовы `update` получили **одинаковый** `dt`. Это и есть смысл fixed timestep: физика всегда считается одинаково.

## 3.4. Компромиссы

| Плюсы | Минусы |
|---|---|
| Стабильная физика | Дороже CPU (update может вызываться чаще, чем render) |
| Предсказуемое поведение | Нужна интерполяция для плавности |
| Легко тестировать | Сложнее реализовать |

## 3.5. Как это улучшит проект

В «Тайне исчезнувшего дива» fixed timestep даст:
- **стабильные таймеры** (например, «через 5 секунд придёт Кузя»);
- **предсказуемую анимацию** (появление текста, подсветка);
- **основу для физики** (в Pygame-версии появятся снаряды и коллизии);
- **тестируемость**: можно симулировать N шагов физики, не запуская игру.

## Для углублённого изучения

- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/) — классическая статья, объясняющая fixed timestep.
- Nystrom R. «Game Programming Patterns»: [Game Loop](https://gameprogrammingpatterns.com/game-loop.html), секция про fixed timestep.
- [Pygame: pygame.time.Clock](https://www.pygame.org/docs/ref/time.html#pygame.time.Clock) — реализация в Pygame.

---
# 4. Variable timestep и его проблемы

## 4.1. Идея

**Variable timestep** — противоположный подход: `update` получает **реальное** время, прошедшее с прошлого кадра:

```python
last_time = now()
while running:
    current_time = now()
    dt = current_time - last_time
    update(dt)
    render()
    last_time = current_time
```

## 4.2. Плюсы

- **Простота.** Не нужен аккумулятор.
- **Точность на быстром железе.** Если FPS высокий, `dt` маленький, движение плавное.
- **Отсутствие двойного счёта.** `update` вызывается ровно один раз за кадр.

## 4.3. Проблемы

**Проблема 1. Нелинейность физики.**

Если `update` использует формулу вида `position += velocity * dt`, при `dt = 0.01` и `dt = 0.1` результат будет **разным** из-за ошибок округления. Игра ведёт себя по-разному на разных машинах.

**Проблема 2. Коллизии на быстрых кадрах.**

Если `dt` большой (низкий FPS), снаряд может «проскочить» сквозь стену: он был перед стеной в кадре N, а в кадре N+1 оказался за ней. Проверка коллизий не сработала.

**Проблема 3. Дрожание при нестабильном FPS.**

Если FPS скачет (например, 60 → 30 → 60), движение будет выглядеть рвано.

**Проблема 4. Невоспроизводимость.**

Тесты становятся недетерминированными: результат зависит от реального времени.

Демонстрация нелинейности:

In [ ]:
def simulate_movement(dt: float, steps: int, speed: float = 100.0) -> float:
    """Симуляция движения: position += speed * dt.

    Возвращает итоговую позицию. При разных dt результаты
    могут отличаться из-за накопления ошибок.
    """
    position = 0.0
    for _ in range(steps):
        position += speed * dt
    return position


# Симулируем 1 секунду движения при разных dt
print("=== Variable timestep: 1 секунда движения ===")
for dt in [0.001, 0.01, 0.1]:
    steps = int(1.0 / dt)
    result = simulate_movement(dt, steps)
    print(f"dt = {dt:.3f} ({steps} шагов): позиция = {result:.4f}")

print("\n=== Fixed timestep: 1 секунда движения ===")
for frame_dt in [0.001, 0.01, 0.1]:
    fixed_dt = 1.0 / 60
    total_time = 1.0
    frames = int(total_time / frame_dt)
    position = 0.0
    accumulator = 0.0
    for _ in range(frames):
        accumulator += frame_dt
        while accumulator >= fixed_dt:
            position += 100.0 * fixed_dt
            accumulator -= fixed_dt
    print(f"frame_dt = {frame_dt:.3f}: позиция = {position:.4f}")

**Вывод:** при variable timestep результат зависит от `dt`. При fixed timestep — всегда одинаковый.

## 4.4. Когда variable timestep допустим

- **Медленные игры** без физики и коллизий (пошаговые стратегии, визуальные новеллы).
- **Игры, где важна плавность**, а не точность (простые аркады).
- **Прототипы** — когда некогда возиться с fixed timestep.

## 4.5. Когда необходим fixed timestep

- **Платформеры** — прыжки должны быть одинаковыми.
- **Файтинги** — кадры важны для соревновательной игры.
- **Сетевые игры** — рассинхрон недопустим.
- **Физические симуляции** — точность критична.

## Для углублённого изучения

- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/)
- [Gaffer On Games: Integration Basics](https://gafferongames.com/post/integration_basics/)
- Nystrom R. «Game Programming Patterns»: [Game Loop](https://gameprogrammingpatterns.com/game-loop.html).

---
# 5. Спираль смерти

## 5.1. Что это

**Спираль смерти** (death spiral) — ситуация, когда игра пытается «догнать» время и попадает в замкнутый цикл, из которого не может выйти.

**Как это происходит:**
1. Компьютер тормозит, `frame_time` растёт.
2. В аккумуляторе накапливается много времени.
3. `update` вызывается много раз, чтобы «догнать» время.
4. Это ещё больше замедляет компьютер.
5. `frame_time` ещё больше растёт.
6. Игра полностью зависает.

## 5.2. Как предотвратить

**Ограничение количества шагов на кадр** (`max_steps`):

```python
while accumulator >= FIXED_DT and steps < MAX_STEPS:
    update(FIXED_DT)
    accumulator -= FIXED_DT
    steps += 1

# Если не успели — выбрасываем остаток времени
if accumulator > MAX_ACCUMULATED:
    accumulator = 0
```

**Идея:** если игра не успевает обработать все шаги, лучше **пропустить время**, чем зависнуть. Игрок заметит небольшое замедление, но игра останется играбельной.

## 5.3. Демонстрация

Ниже — симуляция спирали смерти и её решения через `max_steps`.

In [ ]:
def simulate_death_spiral(frame_times: list[float], max_steps: int | None,
                          fixed_dt: float = 1.0 / 60) -> None:
    """Симуляция игрового цикла с растущим временем кадра.

    Без max_steps — спираль смерти.
    С max_steps — ограничение.
    """
    accumulator = 0.0
    total_updates = 0
    updates_per_frame = []

    for ft in frame_times:
        accumulator += ft
        steps = 0
        while accumulator >= fixed_dt:
            if max_steps is not None and steps >= max_steps:
                accumulator = 0
                break
            accumulator -= fixed_dt
            steps += 1
            total_updates += 1
        updates_per_frame.append(steps)

    label = "С max_steps" if max_steps else "Без max_steps (спираль)"
    print(f"\n{label}:")
    print(f"  Всего update: {total_updates}")
    print(f"  Макс. update за кадр: {max(updates_per_frame)}")
    print(f"  Обновлений за кадр: {updates_per_frame}")


# Эмулируем замедляющийся компьютер
frame_times = [0.016, 0.020, 0.030, 0.050, 0.100, 0.200, 0.500]
print("=== Симуляция спирали смерти ===")
simulate_death_spiral(frame_times, max_steps=None)
simulate_death_spiral(frame_times, max_steps=5)

**Вывод:** без `max_steps` количество update растёт лавинообразно. С `max_steps = 5` мы ограничиваем нагрузку — игра замедлится, но не зависнет.

## 5.4. Как это улучшит проект

Даже в текстовой игре «Тайна исчезнувшего дива» есть риск: если игрок долго не вводит команду, а `frame_time` копится, при следующем вводе игра «прокрутит» много шагов. Ограничение `max_steps` защитит от этого.

## Для углублённого изучения

- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/) — раздел про death spiral.
- Nystrom R. «Game Programming Patterns»: [Game Loop](https://gameprogrammingpatterns.com/game-loop.html).

---
# 6. Интерполяция для плавности

## 6.1. Проблема

При fixed timestep `update` вызывается с частотой 60 Гц, но `render` — с частотой железа. Если железо медленное (30 FPS), между кадрами отрисовки проходит **два** шага физики. Игрок увидит «прыжки»: позиции обновляются раз в 2 кадра.

## 6.2. Решение: интерполяция

**Интерполяция** — это отрисовка объекта **между** двумя состояниями физики. Мы знаем позицию в кадре N и в кадре N+1, а рисуем где-то посередине.

```python
alpha = accumulator / fixed_dt  # 0..1
render_position = prev_position + (current_position - prev_position) * alpha
```

**Зачем:** игра выглядит плавно даже при низком FPS. Это стандарт в современных движках.

## 6.3. Демонстрация

In [ ]:
class InterpolatedMover:
    """Объект с интерполяцией для плавной отрисовки.

    Хранит две позиции: текущую (после update) и предыдущую.
    render_position(alpha) возвращает интерполированную позицию.
    """

    def __init__(self, speed: float = 100.0):
        self.speed = speed
        self.prev_position = 0.0
        self.current_position = 0.0

    def update(self, dt: float) -> None:
        self.prev_position = self.current_position
        self.current_position += self.speed * dt

    def render_position(self, alpha: float) -> float:
        """alpha: 0 — предыдущий кадр, 1 — текущий."""
        return self.prev_position + (self.current_position - self.prev_position) * alpha


# Демонстрация
mover = InterpolatedMover(speed=60.0)
fixed_dt = 1.0 / 60

print("=== Интерполяция при 30 FPS render и 60 FPS update ===")
for render_frame in range(3):
    # Два шага физики за кадр render
    mover.update(fixed_dt)
    pos1 = mover.current_position
    mover.update(fixed_dt)
    pos2 = mover.current_position

    # Интерполируем на 0.5
    interpolated = mover.render_position(0.5)
    print(f"  Кадр {render_frame}: pos1={pos1:.1f}, pos2={pos2:.1f}, "
          f"render={interpolated:.1f}")

## 6.4. Когда нужна интерполяция

- **Платформеры** с плавным движением.
- **Шутеры** с быстрыми снарядами.
- **Стратегии** с камерой.

**Когда не нужна:**
- Пошаговые игры.
- Игры с дискретными позициями (шахматы, карты).
- Текстовая игра «Тайна исчезнувшего дива» — пока не нужна.

## Для углублённого изучения

- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/) — раздел про интерполяцию.
- [Gabriel Gambetta: Fixed Timestep Interpolation](https://www.gabrielgambetta.com/entity-interpolation.html) — практический разбор.

---
# 7. Update Method

## 7.1. Проблема: как обновлять множество объектов

Представьте, что в игре 100 объектов: детектив, див, локации, снаряды, двери, предметы. Каждый должен обновляться каждый кадр. Как это сделать?

**Наивное решение:** цепочка `if isinstance`:

```python
for obj in world.objects:
    if isinstance(obj, Detective):
        obj.update_detective(dt)
    elif isinstance(obj, Div):
        obj.update_div(dt)
    elif isinstance(obj, Bullet):
        obj.update_bullet(dt)
    # ... 20 elif
```

**Проблема:**
- Каждый новый тип объекта = новый `elif` (нарушает OCP).
- Один гигантский метод (нарушает SRP).
- Невозможно тестировать.

## 7.2. Паттерн Update Method

**Update Method** предлагает простое решение: каждый объект имеет метод `update(dt)`. Мир вызывает его единообразно.

```python
for obj in world.objects:
    if obj.active:
        obj.update(dt)
```

**Ключевая идея:** полиморфизм. Мир не знает, какой конкретно объект перед ним. Он вызывает `update`, и каждый объект сам решает, что делать.

## 7.3. Что внутри `update`

У каждого типа объекта своя логика:

- **Детектив:** ничего (управляется игроком).
- **Снаряд:** двигается, проверяет коллизии, исчезает при попадании.
- **Див:** может патрулировать, менять форму по таймеру.
- **Дверь:** анимация открытия.
- **Таймер:** уменьшается, срабатывает при достижении нуля.

## 7.4. Порядок обновления

**Важный вопрос:** в каком порядке вызывать `update`? Если снаряд зависит от позиции игрока — игрок должен обновиться первым. Если игрок зависит от снаряда — наоборот.

**Решения:**
1. **Простой список** — порядок добавления. Работает для большинства случаев.
2. **Приоритеты** — каждый объект имеет `update_priority`.
3. **Фазы** — сначала физика, потом AI, потом анимации.

## 7.5. Реализация в проекте

В проекте «Тайна исчезнувшего дива» все игровые объекты наследуются от `GameObject`, у которого уже есть `update(dt: float)`. Сейчас он реализован как заглушка (`pass`). В этом модуле мы наполним его смыслом.

In [ ]:
# Демонстрация: простой мир с update-методом
from abc import ABC, abstractmethod


class GameObject(ABC):
    """Упрощённая версия из модуля 2."""

    _next_id = 0

    def __init__(self, name: str):
        self._id = GameObject._next_id
        GameObject._next_id += 1
        self._name = name
        self._active = True

    @property
    def name(self) -> str:
        return self._name

    @property
    def active(self) -> bool:
        return self._active

    def deactivate(self) -> None:
        self._active = False

    @abstractmethod
    def update(self, dt: float) -> None:
        ...

    @abstractmethod
    def draw(self) -> str:
        ...


class MovingBullet(GameObject):
    """Снаряд: летит и исчезает через 2 секунды."""

    def __init__(self, name: str, speed: float = 100.0, lifetime: float = 2.0):
        super().__init__(name)
        self.x = 0.0
        self.speed = speed
        self.lifetime = lifetime

    def update(self, dt: float) -> None:
        self.x += self.speed * dt
        self.lifetime -= dt
        if self.lifetime <= 0:
            self.deactivate()

    def draw(self) -> str:
        return f"Bullet at x={self.x:.1f}"


class Timer(GameObject):
    """Таймер: срабатывает через N секунд."""

    def __init__(self, name: str, duration: float):
        super().__init__(name)
        self.duration = duration
        self.fired = False

    def update(self, dt: float) -> None:
        if self.fired:
            return
        self.duration -= dt
        if self.duration <= 0:
            self.fired = True
            self.deactivate()

    def draw(self) -> str:
        return f"Timer({self.name}): {self.duration:.2f}s left"


# Мир: список объектов, все обновляются единообразно
class World:
    def __init__(self):
        self.objects: list[GameObject] = []

    def add(self, obj: GameObject) -> None:
        self.objects.append(obj)

    def update(self, dt: float) -> None:
        for obj in self.objects:
            if obj.active:
                obj.update(dt)

    def draw(self) -> list[str]:
        return [obj.draw() for obj in self.objects if obj.active]


print("=== Update Method: единообразное обновление ===")
world = World()
world.add(MovingBullet("bullet1", speed=50.0, lifetime=1.5))
world.add(Timer("divArrival", duration=1.0))

dt = 1.0 / 60
for frame in range(120):
    world.update(dt)
    if frame % 30 == 0:
        print(f"\nКадр {frame}:")
        for line in world.draw():
            print(f"  {line}")

## 7.6. Как это улучшит проект

**До:** `Game.run()` вызывал логику вручную для каждой команды. Мир не «жил» сам по себе.

**После:** мир обновляет все объекты единообразно. Легко добавить:
- «через 5 секунд придёт Кузя» (Timer);
- «снаряд Плети летит и исчезает» (MovingBullet);
- «свеча догорает и меняет освещение» (Timer + визуальный эффект).

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Update Method](https://gameprogrammingpatterns.com/update-method.html).
- [Refactoring Guru: Polymorphism](https://refactoring.guru/design-patterns/behavioral-patterns).

---
# 8. Component: композиция поведения

## 8.1. Проблема наследования

В модуле 2 мы обсуждали комбинаторный взрыв наследования. Если у персонажа есть:
- 3 типа движения;
- 2 типа оружия;
- 2 типа брони;

Наследование потребует `3 × 2 × 2 = 12` подклассов. Добавление четвёртого параметра — уже 24.

## 8.2. Идея Component

**Component** — это паттерн, при котором объект **собирается из компонентов** — небольших классов, каждый из которых отвечает за одну способность.

Пример: `Detective` состоит из:
- `Position` — координаты;
- `Health` — здоровье;
- `Inventory` — инвентарь;
- `MoveCapability` — способность двигаться;
- `AttackCapability` — способность атаковать.

Каждый компонент — независимая деталь. Комбинируя их, мы получаем любое поведение без создания новых подклассов.

## 8.3. Component vs наследование

| Критерий | Наследование | Компоненты |
|---|---|---|
| Число классов | Экспоненциальный рост | Линейный рост |
| Изменение во время выполнения | Невозможно | Возможно (заменить компонент) |
| Переиспользование | Ограничено | Максимальное |
| Сложность | Растёт с числом параметров | Постоянная |

## 8.4. Реализация компонента

Простейший компонент — это dataclass:

```python
@dataclass
class Position:
    x: float = 0.0
    y: float = 0.0
```

Объект **содержит** компоненты и делегирует им поведение.

## 8.5. Преимущества

1. **Композиция вместо наследования.** Объект собирается из деталей.
2. **Динамичность.** Можно добавить или удалить компонент во время игры.
3. **Переиспользование.** Один компонент `Health` используется у детектива, дива и врагов.
4. **Тестируемость.** Каждый компонент тестируется отдельно.

## 8.6. Недостатки

1. **Косвенность.** Не сразу видно, что умеет объект — нужно смотреть состав.
2. **Связность между компонентами.** Компонентам иногда нужно общаться — например, `Movement` зависит от `Position`.
3. **Больше классов.** Простая игра может не нуждаться в компонентах.

## Для углублённого изучения

- Nystrom R. «Game Programming Patterns»: [Component](https://gameprogrammingpatterns.com/component.html).
- [Wikipedia: Entity Component System](https://en.wikipedia.org/wiki/Entity_component_system).
- [Unity Manual: Components](https://docs.unity3d.com/Manual/Components.html) — пример компонентного подхода в реальном движке.

---
# 9. Component в стиле dataclass

## 9.1. Компоненты для «Тайны исчезнувшего дива»

Разложим нашего `Detective` на компоненты:
- `Position` — где находится;
- `Health` — энергия/здоровье;
- `Inventory` — найденные улики;
- `MoveSpeed` — скорость передвижения.

А `Location`:
- `ClueStorage` — список улик;
- `CharacterList` — список персонажей.

## 9.2. Реализация через dataclass

Каждый компонент — dataclass. Объект содержит их как атрибуты.

In [ ]:
from dataclasses import dataclass, field


# === Компоненты ===

@dataclass
class Position:
    """Компонент: координаты объекта."""
    x: float = 0.0
    y: float = 0.0

    def move(self, dx: float, dy: float) -> None:
        self.x += dx
        self.y += dy


@dataclass
class Health:
    """Компонент: здоровье/энергия объекта."""
    current: int = 100
    maximum: int = 100

    @property
    def is_alive(self) -> bool:
        return self.current > 0

    def damage(self, amount: int) -> None:
        self.current = max(0, self.current - amount)

    def heal(self, amount: int) -> None:
        self.current = min(self.maximum, self.current + amount)


@dataclass
class Inventory:
    """Компонент: инвентарь объекта."""
    items: list = field(default_factory=list)

    def add(self, item) -> None:
        if item not in self.items:
            self.items.append(item)

    @property
    def size(self) -> int:
        return len(self.items)


@dataclass
class MoveSpeed:
    """Компонент: скорость движения."""
    speed: float = 100.0


# === Объект, собранный из компонентов ===

class Entity:
    """Сущность: контейнер для компонентов.

    Не имеет собственной логики — только хранение компонентов.
    Поведение определяется составом компонентов.
    """

    def __init__(self, name: str):
        self.name = name
        self._components: dict[type, object] = {}

    def add_component(self, component: object) -> None:
        self._components[type(component)] = component

    def get(self, component_type: type):
        return self._components.get(component_type)

    def has(self, component_type: type) -> bool:
        return component_type in self._components

    def remove(self, component_type: type) -> None:
        self._components.pop(component_type, None)

    def __repr__(self) -> str:
        types = ", ".join(t.__name__ for t in self._components)
        return f"Entity({self.name}: [{types}])"


# === Использование ===
detective = Entity("Аверин")
detective.add_component(Position(0, 0))
detective.add_component(Health(100, 100))
detective.add_component(Inventory())
detective.add_component(MoveSpeed(50.0))

print(detective)

# Работа с компонентами
pos = detective.get(Position)
pos.move(10, 20)
print(f"Position: ({pos.x}, {pos.y})")

hp = detective.get(Health)
hp.damage(30)
print(f"Health: {hp.current}/{hp.maximum}")

inv = detective.get(Inventory)
inv.add("Записка с планом Пустоши")
print(f"Inventory: {inv.items}")

# Другой объект — другая комбинация компонентов
door = Entity("Дверь")
door.add_component(Position(5, 5))
door.add_component(Health(50, 50))  # у двери есть прочность
print(f"\n{door}")

## 9.3. Системы и компоненты

Компоненты хранят **данные**. Логика обработки — в **системах**. Это переход к ECS.

**Пример:** система движения проходит по всем сущностям с `Position` и `MoveSpeed` и обновляет позиции:

```python
class MovementSystem:
    def update(self, entities: list[Entity], dt: float) -> None:
        for entity in entities:
            pos = entity.get(Position)
            speed = entity.get(MoveSpeed)
            if pos and speed:
                pos.x += speed.speed * dt
```

## 9.4. Преимущества этого подхода

- **Логика отделена от данных.** Компоненты — только поля. Системы — только поведение.
- **Легко добавлять новые типы.** Новый компонент = новый dataclass. Новая система = новый класс.
- **Оптимизация.** Системы могут работать с массивами компонентов одного типа (кэш-дружественно).

## 9.5. Как это улучшит проект

**До:** каждая сущность — отдельный класс с логикой.

**После:** сущность = набор компонентов. Логика — в системах.

Это позволит:
- добавлять новые типы сущностей без новых классов;
- переиспользовать компоненты (`Health` у детектива, двери, врага);
- тестировать системы с mock-компонентами.

## Для углублённого изучения

- [GitHub: ECS FAQ](https://github.com/SanderMertens/ecs-faq) — большой FAQ по ECS.
- [Adam Martin: Entity Systems are the future of MMOG development](https://t-machine.org/index.php/2007/09/03/entity-systems-are-the-future-of-mmog-development-part-1/) — историческая статья, объясняющая ECS.

---
# 10. Переход к ECS

## 10.1. Что такое ECS

**ECS** (Entity-Component-System) — это эволюция Component-паттерна. Три части:

- **Entity** — идентификатор (число). Не объект, а ссылка.
- **Component** — чистые данные (без методов).
- **System** — чистое поведение (без данных).

## 10.2. Отличие от Component

| Аспект | Component (наш подход) | ECS |
|---|---|---|
| Entity | Объект с компонентами | Число-идентификатор |
| Component | Может иметь методы | Только данные |
| System | Не формализована | Явный класс |
| Хранение | В объекте | В массивах по типу (кэш) |

## 10.3. Пример ECS-системы

```python
class MovementSystem:
    def update(self, world, dt):
        for entity_id in world.query(Position, Velocity):
            pos = world.get_component(entity_id, Position)
            vel = world.get_component(entity_id, Velocity)
            pos.x += vel.dx * dt
            pos.y += vel.dy * dt
```

**Ключевое:** система выбирает только те сущности, у которых есть **оба** компонента.

## 10.4. Когда использовать ECS

**Стоит использовать:**
- Много объектов (тысячи);
- Много разных комбинаций поведения;
- Нужна производительность (кэш-дружественность).

**Не стоит:**
- Маленькие игры (10–50 объектов);
- Простые игры без динамического поведения;
- Наша «Тайна исчезнувшего дива» — для неё ECS избыточен.

## 10.5. Что мы используем в курсе

В курсе мы остаёмся на **Component-паттерне** (не ECS). Причины:
- Наша игра — текстовая, объектов немного.
- Component даёт 80% преимуществ ECS при меньшей сложности.
- ECS — тема для больших команд и движков.

**Но:** в модуле 7 мы посмотрим на ECS **концептуально** — как на современный подход.

## Для углублённого изучения

- [GitHub: ECS FAQ](https://github.com/SanderMertens/ecs-faq).
- [Bevy Engine: ECS](https://bevyengine.org/learn/book/getting-started/ecs/) — реализация в игровом движке Rust.
- [Unity DOTS](https://unity.com/dots) — ECS в Unity.

---
# 11. Интеграция с Pygame

## 11.1. Pygame как платформа

**Pygame** — библиотека для 2D-игр на Python. Она предоставляет:
- окно и рендеринг;
- обработку событий (клавиатура, мышь);
- спрайты и группы;
- звук;
- таймеры и clock.

## 11.2. Игровой цикл в Pygame

Pygame использует `pygame.time.Clock` для управления FPS:

```python
import pygame

pygame.init()
screen = pygame.display.set_mode((800, 600))
clock = pygame.time.Clock()

running = True
while running:
    # 1. Обработка событий
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # 2. Обновление
    dt = clock.tick(60) / 1000.0  # мс → сек

    # 3. Отрисовка
    screen.fill((0, 0, 0))
    # ... рисование
    pygame.display.flip()

pygame.quit()
```

## 11.3. Компоненты и Pygame Sprite

`pygame.sprite.Sprite` — это встроенный класс Pygame, который можно рассматривать как «полу-компонент»: у него есть `image`, `rect`, `update()`, `draw()`.

Мы можем комбинировать Pygame Sprite с нашими компонентами:

```python
class PlayerSprite(pygame.sprite.Sprite):
    def __init__(self, entity: Entity):
        super().__init__()
        self.entity = entity
        self.image = pygame.Surface((32, 32))
        self.rect = self.image.get_rect()

    def update(self, dt):
        pos = self.entity.get(Position)
        self.rect.x = pos.x
        self.rect.y = pos.y
```

## 11.4. Для нашего проекта

«Тайна исчезнувшего дива» останется **текстовой** в этом модуле. Pygame подключим в модуле 7, когда будем изучать ECS и слоистую архитектуру. Но **игровой цикл** и **компоненты** — уже здесь.

## Для углублённого изучения

- [Pygame documentation](https://www.pygame.org/docs/) — официальная документация.
- [Real Python: Pygame Tutorial](https://realpython.com/pygame-a-primer/) — введение.
- [Pygame: pygame.time.Clock](https://www.pygame.org/docs/ref/time.html#pygame.time.Clock) — управление FPS.

---
# 12. Итоги и литература

## Что мы изучили

| Паттерн | Назначение | Ключевая идея |
|---|---|---|
| **Game Loop** | Постоянное обновление и отрисовка | Цикл, работающий с временем |
| **Fixed timestep** | Стабильная физика | Аккумулятор + постоянный `dt` |
| **Death spiral** | Защита от зависания | Ограничение `max_steps` |
| **Интерполяция** | Плавность при низком FPS | Отрисовка между кадрами |
| **Update Method** | Единообразное обновление объектов | Полиморфный `update(dt)` |
| **Component** | Композиция поведения | Объект = контейнер компонентов |
| **ECS** | Эволюция Component | Entity-Component-System |

## Что появится в проекте

- **Игровой цикл** с фиксированным шагом `dt`.
- **Аккумулятор** и защита от спирали смерти.
- **Компоненты** `Position`, `Health`, `Inventory`, `MoveSpeed`.
- **Entity** как контейнер компонентов.
- **Системы**, работающие с компонентами.

## Практические рекомендации

1. **Используйте fixed timestep**, если в игре есть физика или таймеры.
2. **Ограничивайте `max_steps`**, чтобы избежать спирали смерти.
3. **Разделяйте update и render** — это упрощает тестирование.
4. **Компонуйте поведение через компоненты**, а не через наследование.
5. **Каждый компонент — dataclass**, если возможно.
6. **Каждая система — отдельный класс** с одним методом `update`.

## Литература

**Базовая:**
- Nystrom R. «Game Programming Patterns»: [Game Loop](https://gameprogrammingpatterns.com/game-loop.html), [Update Method](https://gameprogrammingpatterns.com/update-method.html), [Component](https://gameprogrammingpatterns.com/component.html).
- [Pygame documentation](https://www.pygame.org/docs/).
- [Real Python: Pygame Tutorial](https://realpython.com/pygame-a-primer/).

**Углублённая:**
- [Gaffer On Games: Fix Your Timestep!](https://gafferongames.com/post/fix_your_timestep/) — классическая статья про fixed timestep.
- [Gaffer On Games: Integration Basics](https://gafferongames.com/post/integration_basics/).
- [GitHub: ECS FAQ](https://github.com/SanderMertens/ecs-faq) — большой FAQ по ECS.
- [Adam Martin: Entity Systems](https://t-machine.org/index.php/2007/09/03/entity-systems-are-the-future-of-mmog-development-part-1/).
- [Bevy Engine: ECS](https://bevyengine.org/learn/book/getting-started/ecs/) — реализация в Rust.

## Что дальше

В модуле 5 мы изучим:
- **State** — управление состояниями игры (меню, расследование, бой, диалог).
- **Strategy** — взаимозаменяемые алгоритмы (поведение Кузи, типы атак).
- **Command** — инкапсуляция действий игрока с поддержкой undo.

Все три паттерна напрямую строятся на игровом цикле и компонентах, которые мы изучили в этом модуле.